In [ ]:
set.seed(1)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(tidyverse)
library(patchwork)
library(harmony)
library(readxl)
library(viridis)

In [ ]:
date <- "2026-06-06_develop_all"

In [ ]:
data_dir <- "/project/work/Data/"
plot_dir <- "/project/work/Plot/"

# 准备颜色

In [ ]:
library(RColorBrewer)

In [ ]:
cols.palette1 <- c(brewer.pal(9, "Set1"),brewer.pal(8, "Dark2"),brewer.pal(8, "Set2"),brewer.pal(8, "Accent"))

## age colors
cols.age <- c(cols.palette1[18:29],cols.palette1[6],cols.palette1[1:5],cols.palette1[8])
names(cols.age) <- c('70days','79days','87days','91days','100days','103days',
                     '116days','137days','141days','142days','162days','165days',
                     '21years','25years',
                     '67years','70years','75years','86years')

In [ ]:
cols.palette2 <- c(brewer.pal(12, "Paired"),brewer.pal(8, "Set2"),brewer.pal(8, "Dark2"),brewer.pal(9, "Set1"))

## celltype colors
cols.celltype <- c(cols.palette2[1:8],cols.palette2[13:14])
names(cols.celltype) <- c('Rod', 'Cone', 'AC', 'BC', 'HC', 'MG', 'RGC', 'RPC','Micro','Astro')

## tissue colors
cols.tissue <- cols.palette2[9:11]
names(cols.tissue) <- c('fovea', 'macula', 'peripheral')

# 准备数据

In [ ]:
obj <- readRDS('/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds')
obj

# umap

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
p_umap <- DimPlot(obj, reduction = "umap.harmony", group.by = "celltype", label = TRUE , cols = cols.celltype, raster=TRUE)
p_umap

In [ ]:
pdf(paste0(plot_dir,date,"_umap_celltype",".pdf"), width = 8, height = 8)
  print(p_umap)
dev.off()
paste0(plot_dir,date,"_umap_celltype",".pdf")

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
p_umap <- DimPlot(obj, reduction = "umap.harmony", group.by = "donor_age", label = FALSE , cols = cols.age, raster=TRUE)
p_umap

In [ ]:
pdf(paste0(plot_dir,date,"_umap_age",".pdf"), width = 8, height = 8)
  print(p_umap)
dev.off()
paste0(plot_dir,date,"_umap_age",".pdf")

In [ ]:
cols.region <- c('#6A3D9A','#E69F00')
names(cols.region) <- c('central', 'peripheral')

In [ ]:
obj$region <- as.character(obj$tissue)

In [ ]:
obj$region[obj$tissue %in% c('fovea','macula')] = 'central'

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
p_umap <- DimPlot(obj, reduction = "umap.harmony", group.by = "region", label = FALSE , cols = cols.region, raster=TRUE)
p_umap

In [ ]:
pdf(paste0(plot_dir,date,"_umap_region",".pdf"), width = 8, height = 8)
  print(p_umap)
dev.off()
paste0(plot_dir,date,"_umap_region",".pdf")

In [ ]:
saveRDS(CONTACT_EMAIL, "/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno_meta.rds")

# marker

In [ ]:
obj$celltype <- factor(obj$celltype,level = c('Rod','Cone','AC','BC','HC','MG','RGC','RPC'))

In [ ]:
use.markers <- c(
  "RHO", "CNGA1", "SAG", "GNAT1",    #"PDE6A",            # Rod
  "OPN1MW", "OPN1SW", "PDE6H",  "GUCA1C",    #"ARR3",      # Cone
  "GAD1", "SLC6A9", "GAD2", #"TFAP2B", "TFAP2A",          # AC
  "VSX1", "GRM6", "TRPM1", #"PRKCA", "CABP5",             # BC
  "ONECUT1", "ONECUT2", "LHX1", #"CALB1", "PROX1",        # HC
  "RLBP1", "GLUL", "APOE", #"SOX9", "DKK3",                 # MG
  "POU4F1", "POU4F2", "NEFL", #"RBPMS",# "ATOH7", "SNCG",  # RGC
  "SOX2", "HES1", "MKI67"#, "PAX6", "RAX"                # RPC  
)

length(unique(use.markers))

In [ ]:
use.types <- c(
  # "NRPC", "NRPC", "NRPC", "NRPC", "NRPC",
  # "PRPC", "PRPC", "PRPC", "PRPC",
  "Rod", "Rod", "Rod", "Rod", #"Rod",
  "Cone", "Cone", "Cone", "Cone", #"Cone",
  "AC", "AC", "AC", #"AC", "AC",
  "BC", "BC", "BC", #"BC", "BC",
  "HC", "HC", "HC",# "HC", "HC",
  "MG", "MG", "MG", #"MG", "MG",
  "RGC", "RGC", "RGC", #"RGC",# "RGC", "RGC",
  "RPC", "RPC", "RPC"#, "RPC", "RPC" #,"RPC"
)

length(use.types)

In [ ]:
use.markers.df <- data.frame(features.plot = use.markers, type = use.types)

In [ ]:
p <- DotPlot(obj, features = use.markers, group.by = "celltype")+
scale_colour_gradient2(low = "black", mid = "grey", high = "red")+ 
theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"))

# p

In [ ]:
use.data <- p$data

In [ ]:
use.data <- merge(use.data, use.markers.df, by = "features.plot")

In [ ]:
use.data$features.plot <- factor(use.data$features.plot, levels = use.markers)

In [ ]:
use.data$type <- factor(use.data$type, levels = unique(use.types))

In [ ]:
p1 <- ggplot(use.data, aes(x = features.plot, y = id)) +
  geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +  # 使用点大小和颜色表示信息
  scale_size(range = c(0.1, 5)) +  # 调整点的大小范围
  # scale_color_gradient2(low = "black", mid = "grey", high = "red") +  # 定义颜色梯度
  scale_color_gradientn(
    colours = c("#313695", "#762A83", "#D6604D", "#FFFF00"),
    name = "Scaled\nExpression"
  )+  # 定义颜色梯度
  facet_wrap(~ type, scales = "free_x", nrow = 1) +  # 根据类型分面
  theme_classic() +
  theme(text = element_text(color = "black"),
    axis.text.y = element_text(color = "black", size = 12),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 10),
    legend.title = element_text(size = 10),  # 图例标题字号及下方间距
    strip.text = element_text(size = 11)  # 调整分面标签字体大小
  ) +
  labs(x = "Features", y = "Cell Type", size = "Percentag\n Expressed", color = "Scaled\nExpression")

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 4)
p1

In [ ]:
# ===== 追加(2026-09-14)：celltype 顺序反转 RPC->Rod（翻转 y 轴），另存新图，不覆盖原图 =====
# 复用内存中已有的 p1（obj -> p$data -> use.data 生成），仅翻转 y 轴显示顺序，其它不动
p1_rev <- p1 + scale_y_discrete(limits = rev)

options(repr.plot.width = 12, repr.plot.height = 4)
p1_rev

pdf(paste0(plot_dir, date, "_marker_celltypeRev", ".pdf"), width = 12, height = 4)
  print(p1_rev)
dev.off()
paste0(plot_dir, date, "_marker_celltypeRev", ".pdf")